# 04. 법률 데이터 4종 멀티 인덱스 구축

판례·헌재결정례·법제처해석례·행정심판재결례 4개 데이터 소스에 대해
각각 최적화된 스키마로 AI Search 인덱스를 생성하고 데이터를 투입합니다.

## 비용 최적화 전략
| 인덱스 | 임베딩 대상 필드 | 비임베딩 (키워드 검색) | 필터 메타데이터 |
|--------|-----------------|----------------------|----------------|
| `prec-court-index` | 판시사항 + 판결요지 | 사건명, 전문 | 법원명, 심급, 사건종류, 관련법령 |
| `const-court-index` | 판시사항 + 결정요지 | 사건명, 전문 | 결정유형, 관련법령, 주제어 |
| `legis-interp-index` | 질의요지 + 회답 | 안건명, 이유 | 안건종류, 관련법령, 주제어 |
| `admin-appeal-index` | 주문 + 이유요약 | 사건명, 전문 | 재결유형, 심판위원회, 관련법령 |

> 전문(fullText) 전체를 벡터화하지 않고 **요지·핵심 필드만 임베딩**하여 비용을 절약합니다.

## 1. 환경 설정

In [ ]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

print(f"Search Endpoint: {os.environ.get('AZURE_SEARCH_SERVICE_ENDPOINT')}")
print(f"OpenAI Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT')}")
print(f"Admin Key 설정: {'✅' if os.environ.get('AZURE_SEARCH_ADMIN_KEY') else '❌'}")

## 2. 4개 인덱스 생성

`LegalIndexManager`를 사용하여 4개 인덱스를 한번에 생성합니다.
각 인덱스는 text-embedding-3-large (3072차원) 벡터 검색을 지원합니다.

In [ ]:
from src.search.legal_indexes import LegalIndexManager, PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

mgr = LegalIndexManager(
    endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'],
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

# 4개 인덱스 한번에 생성
mgr.create_all_indexes(dimensions=3072)

## 3. 실제 데이터 수집 (law.go.kr 크롤러)

웹 스크래핑 방식으로 실제 law.go.kr 데이터를 수집합니다 (IP 등록 불필요).  
크롤러 출력의 한국어 필드명을 AI Search 인덱스 스키마(영문 필드명)로 매핑합니다.

| 크롤러 | 수집 | 주요 매핑 |
|--------|------|-----------|
| `LawPrecedentCrawler` | 판례 | 사건명→caseName, 판시사항→holdings, 판결요지→summary |
| `HunjaeCrawler` | 헌재결정례 | 사건명→caseName, 판시사항→holdings, 결정요지→summary |
| `ExpCrawler` | 법제처해석례 | 제목→title, 질의요지→querySummary, 회답→reply |
| `AdmRulCrawler` | 행정심판재결례 | 사건명→caseName, 재결요지→reasonSummary, 주문→order |

In [ ]:
from src.crawler.precedent_crawler import (
    LawPrecedentCrawler, HunjaeCrawler, ExpCrawler, AdmRulCrawler
)
from datetime import datetime, timezone

def _parse_date(s: str) -> str:
    """'2003. 12. 12.' → '2003-12-12T00:00:00Z'"""
    try:
        parts = [p.strip() for p in s.strip().rstrip('.').split('.') if p.strip()]
        if len(parts) >= 3:
            return f"{parts[0]}-{int(parts[1]):02d}-{int(parts[2]):02d}T00:00:00Z"
    except Exception:
        pass
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

_NOW = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def map_prec(doc: dict) -> dict:
    court = doc.get("법원명", "")
    level = "3심" if "대법원" in court else "2심" if any(x in court for x in ["고등법원", "고법"]) else "1심"
    return {
        "id": f"prec_{doc['seq']}",
        "courtName": court,
        "caseNumber": doc.get("사건번호", ""),
        "judgmentDate": _parse_date(doc.get("선고일자", "")),
        "courtLevel": level,
        "caseType": "",
        "status": "확정",
        "relatedLaws": doc.get("참조조문", ""),
        "keywords": "",
        "registrationDate": _NOW,
        "caseName": doc.get("사건명", ""),
        "holdings": doc.get("판시사항", ""),
        "summary": doc.get("판결요지", ""),
        "fullText": doc.get("전문", ""),
    }

def map_detc(doc: dict) -> dict:
    return {
        "id": f"detc_{doc['seq']}",
        "caseNumber": doc.get("사건번호", ""),
        "decisionDate": _parse_date(doc.get("결정일자", "")),
        "decisionType": "",
        "relatedLaws": doc.get("심판대상조문", "") or doc.get("참조조문", ""),
        "keywords": "",
        "fiscalYear": "",
        "registrationDate": _NOW,
        "caseName": doc.get("사건명", ""),
        "holdings": doc.get("판시사항", ""),
        "summary": doc.get("결정요지", ""),
        "fullText": doc.get("전문", ""),
    }

def map_expc(doc: dict) -> dict:
    return {
        "id": f"expc_{doc['seq']}",
        "docNumber": doc.get("문서번호", ""),
        "replyDate": _parse_date(doc.get("회시일자", "")),
        "interpType": "법령해석",
        "relatedLaws": "",
        "keywords": "",
        "registrationDate": _NOW,
        "title": doc.get("제목", ""),
        "querySummary": doc.get("질의요지", ""),
        "reply": doc.get("회답", ""),
        "reason": doc.get("이유", ""),
    }

def map_admrul(doc: dict) -> dict:
    return {
        "id": f"admrul_{doc['seq']}",
        "caseNumber": doc.get("사건번호", ""),
        "decisionDate": _parse_date(doc.get("재결일자", "")),
        "decisionType": doc.get("재결결과", ""),
        "relatedLaws": "",
        "keywords": "",
        "committee": doc.get("재결기관", ""),
        "registrationDate": _NOW,
        "caseName": doc.get("사건명", ""),
        "order": doc.get("주문", ""),
        "reasonSummary": doc.get("재결요지", ""),
        "fullText": doc.get("이유", ""),
    }

# 크롤러 설정: (크롤러, seq 필드명, 매핑 함수)
CRAWLER_MAP = {
    PREC_INDEX:  (LawPrecedentCrawler(), "precSeq",  map_prec),
    CONST_INDEX: (HunjaeCrawler(),       "detcSeq",  map_detc),
    INTERP_INDEX:(ExpCrawler(),          "expcSeq",  map_expc),
    ADMIN_INDEX: (AdmRulCrawler(),       "deccSeq",  map_admrul),
}

SAMPLE_SIZE = 5  # 인덱스당 수집 건수 (테스트 목적)

data_map = {}
for idx_name, (crawler, seq_field, mapper) in CRAWLER_MAP.items():
    items = list(crawler.iter_list(max_pages=1))[:SAMPLE_SIZE]
    docs = []
    for it in items:
        raw = crawler.get_detail(it[seq_field])
        if raw:
            docs.append(mapper(raw))
    data_map[idx_name] = docs
    print(f"[{idx_name}] {len(docs)}건 수집")

print(f"\n총 {sum(len(d) for d in data_map.values())}건")

## 4. 선택적 임베딩 생성

**비용 절약 핵심**: 전체 문서가 아닌 인덱스별 핵심 필드만 임베딩합니다.

```
판례: holdings + summary (판시사항 + 판결요지)
헌재: holdings + summary (판시사항 + 결정요지)  
해석례: querySummary + reply (질의요지 + 회답)
재결례: order + reasonSummary (주문 + 이유요약)
```

In [ ]:
from src.preprocessing.embedding import EmbeddingGenerator

embedder = EmbeddingGenerator(
    endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    deployment=os.environ.get('AZURE_OPENAI_EMBEDDING_DEPLOYMENT', 'text-embedding-3-large'),
)

# 인덱스별로 핵심 필드만 추출하여 임베딩 생성
for idx_name, docs in data_map.items():
    texts_to_embed = [mgr.get_embedding_text(idx_name, doc) for doc in docs]
    embeddings = embedder.generate_batch(texts_to_embed)

    for doc, emb in zip(docs, embeddings):
        doc["summaryEmbedding"] = emb

    # 임베딩 대상 필드와 길이 출력
    fields = mgr.EMBEDDING_FIELDS[idx_name]
    avg_len = sum(len(t) for t in texts_to_embed) // max(len(texts_to_embed), 1)
    print(f"[{idx_name}] {len(docs)}건 × 임베딩 필드={fields} (평균 {avg_len}자)")

print(f"\n임베딩 차원: {len(docs[0]['summaryEmbedding'])}차원")

## 5. 인덱스 업로드

In [ ]:
for idx_name, docs in data_map.items():
    mgr.upload_documents(idx_name, docs)

## 6. 인덱스 통계 확인

In [ ]:
import time
time.sleep(3)  # 인덱싱 반영 대기

for s in mgr.get_all_stats():
    print(f"  {s['index_name']}: {s['document_count']}건")

print("\n✅ 4개 인덱스 구축 완료!")

## 7. 인덱스별 테스트 검색

In [ ]:
# 인덱스별 테스트 검색
test_queries = {
    PREC_INDEX: "의제배당 주식소각 과세",
    CONST_INDEX: "표현의 자유 인터넷",
    INTERP_INDEX: "개인정보 가명정보 제3자 제공",
    ADMIN_INDEX: "건축허가 거부 취소",
}

for idx_name, query in test_queries.items():
    print(f"\n{'='*60}")
    print(f"[{idx_name}] 검색어: '{query}'")
    print(f"{'='*60}")

    embedding = embedder.generate(query)
    results = mgr.search(idx_name, query=query, embedding=embedding, top=3)

    for i, r in enumerate(results, 1):
        # 인덱스별 대표 필드를 출력
        title = r.get("caseName") or r.get("title", "N/A")
        print(f"  {i}. [{r['score']:.4f}] {title[:80]}")

---
다음 단계: [05-multi-index-search.ipynb](05-multi-index-search.ipynb) - 멀티 인덱스 통합 검색 및 Cross-Index RAG